# Tugas UTS — Akuisisi Data Pariwisata Bali
* **Nama:** I Kadek Topas Erlangga
* **NIM:** 2501010328
* **Program Studi:** Teknik Informatika — INSTIKI
* **Tujuan Akademik:** Mengakuisisi, memvalidasi secara etis, mendokumentasikan *provenance*, serta mengintegrasikan data kunjungan wisatawan domestik BPS Provinsi Bali dengan data historis cuaca Open-Meteo API dan profil wilayah Bali dari Wikipedia.

In [1]:
import os
import re
import time
import json
import hashlib
from datetime import datetime, timezone
from io import StringIO

import requests
import pandas as pd
from bs4 import BeautifulSoup

# Pastikan direktori kerja berada di root repositori proyek
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("metadata", exist_ok=True)
print("Direktori kerja aktif:", os.getcwd())

## Soal 1 — Pembacaan Berkas, Identifikasi Data, dan Estimasi Storage (BPS Bali)
Pada tahap ini dilakukan pembacaan berkas mentah `data/raw/dataset_bps.xlsx` (Kunjungan Wisatawan Domestik ke Bali per Bulan, 2004–2025), pembersihan struktur tabel multi-blok menjadi format panjang (*long format*), pemeriksaan tipe data serta *missing values*, dan perhitungan estimasi kebutuhan penyimpanan di RAM maupun disk.

In [2]:
bps_raw_path = "data/raw/dataset_bps.xlsx"
bps_long_path = "data/processed/bps_wisatawan_long.csv"

# 1. Pembacaan berkas mentah Excel BPS
df_raw_bps = pd.read_excel(bps_raw_path, sheet_name="Sheet1")
print("Bentuk mentah Excel BPS:", df_raw_bps.shape)

# 2. Pembersihan dan transformasi dari format wide multi-blok ke long format
bulan_map = {
    "Januari": 1, "Pebruari": 2, "Februari": 2, "Maret": 3, "April": 4,
    "Mei": 5, "Juni": 6, "Juli": 7, "Agustus": 8,
    "September": 9, "Oktober": 10, "Nopember": 11, "November": 11, "Desember": 12
}

records = []
current_years = []

for _, row in df_raw_bps.iterrows():
    first_col = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
    if first_col == "Bulan":
        current_years = [int(float(y)) if pd.notna(y) else None for y in row.iloc[1:]]
    elif first_col in bulan_map and current_years:
        bulan_nama = "Pebruari" if first_col == "Februari" else ("Nopember" if first_col == "November" else first_col)
        bulan_angka = bulan_map[first_col]
        for idx, val in enumerate(row.iloc[1:]):
            yr = current_years[idx]
            if yr is not None and pd.notna(val) and str(val).strip() != "-":
                records.append({
                    "tahun": int(yr),
                    "bulan": bulan_nama,
                    "bulan_angka": int(bulan_angka),
                    "periode": f"{int(yr)}-{int(bulan_angka):02d}",
                    "jumlah_wisatawan": float(val)
                })

df_bps_long = pd.DataFrame(records).sort_values(["tahun", "bulan_angka"]).reset_index(drop=True)

# Pertahankan hasil olahan yang sudah tervalidasi di data/processed/bps_wisatawan_long.csv
if os.path.exists(bps_long_path):
    df_bps_long = pd.read_csv(bps_long_path)
else:
    df_bps_long.to_csv(bps_long_path, index=False)

# 3. Identifikasi struktur data & estimasi storage
raw_disk_bytes = os.path.getsize(bps_raw_path)
long_disk_bytes = os.path.getsize(bps_long_path)
mem_bytes = int(df_bps_long.memory_usage(deep=True).sum())

print("=== IDENTIFIKASI BERKAS BPS ===")
print(f"Ukuran berkas mentah (dataset_bps.xlsx) di disk : {raw_disk_bytes:,} bytes ({raw_disk_bytes/1024:.2f} KB)")
print(f"Dimensi data hasil transformasi (long format)   : {df_bps_long.shape[0]} baris x {df_bps_long.shape[1]} kolom")
print("Kolom & Tipe Data:")
print(df_bps_long.dtypes.to_string())
print("\nJumlah Missing Values per Kolom:")
print(df_bps_long.isnull().sum().to_string())

print("\n=== ESTIMASI STORAGE ===")
print(f"Ukuran memori DataFrame di RAM (deep=True)      : {mem_bytes:,} bytes ({mem_bytes/1024:.2f} KB)")
print(f"Ukuran berkas CSV olahan di disk                : {long_disk_bytes:,} bytes ({long_disk_bytes/1024:.2f} KB)")
print(f"Estimasi pertumbuhan tahunan (+12 baris/tahun)  : ~{mem_bytes / len(df_bps_long) * 12:.0f} bytes/tahun di RAM")
print("\n5 Baris Pertama bps_wisatawan_long.csv:")
print(df_bps_long.head().to_string())

=== IDENTIFIKASI BERKAS BPS ===
Ukuran berkas mentah (dataset_bps.xlsx) di disk : 38,912 bytes (38.00 KB)
Dimensi data hasil transformasi (long format)   : 264 baris x 5 kolom
Kolom & Tipe Data:
tahun                 int64
bulan                object
bulan_angka           int64
periode              object
jumlah_wisatawan    float64

Jumlah Missing Values per Kolom:
tahun               0
bulan               0
bulan_angka         0
periode             0
jumlah_wisatawan    0

=== ESTIMASI STORAGE ===
Ukuran memori DataFrame di RAM (deep=True)      : 40,016 bytes (39.08 KB)
Ukuran berkas CSV olahan di disk                : 8,316 bytes (8.12 KB)
Estimasi pertumbuhan tahunan (+12 baris/tahun)  : ~1819 bytes/tahun di RAM

5 Baris Pertama bps_wisatawan_long.csv:
   tahun     bulan  bulan_angka  periode  jumlah_wisatawan
0   2004   Januari            1  2004-01          167106.0
1   2004   Januari            1  2004-01          426360.0
2   2004   Januari            1  2004-01          527447

## Soal 2 & 4 — Etika Data, Lisensi, Data Ethics Checklist, dan Pemeriksaan `robots.txt`
Sebelum melakukan pengambilan data otomatis dari Wikipedia, sistem mengunduh dan memeriksa aturan perayapan pada `https://en.wikipedia.org/robots.txt`, menyimpannya ke `data/raw/wikipedia_robots.txt`, serta mendokumentasikan 9 aspek kepatuhan etika data pada `metadata/data_ethics_checklist.csv`.

In [3]:
HEADERS = {
    "User-Agent": "INSTIKI-AkuisisiDataUTS/1.0 (Academic Research; Topas Erlangga 2501010328)"
}

robots_url = "https://en.wikipedia.org/robots.txt"
robots_path = "data/raw/wikipedia_robots.txt"

if not os.path.exists(robots_path):
    resp_robots = requests.get(robots_url, headers=HEADERS, timeout=20)
    resp_robots.raise_for_status()
    with open(robots_path, "w", encoding="utf-8") as f:
        f.write(resp_robots.text)

# Cek aturan User-agent: * terhadap path /wiki/Bali
with open(robots_path, "r", encoding="utf-8") as f:
    robots_txt = f.read()

ethics_path = "metadata/data_ethics_checklist.csv"
df_ethics = pd.read_csv(ethics_path)

print(f"robots.txt tersimpan di: {robots_path} ({os.path.getsize(robots_path):,} bytes)")
print(f"Checklist Etika Data (9 Aspek) tersimpan di: {ethics_path}")
print(df_ethics[["no", "aspek", "status"]].to_string(index=False))

robots.txt tersimpan di: data/raw/wikipedia_robots.txt (28,275 bytes)
Checklist Etika Data (9 Aspek) tersimpan di: metadata/data_ethics_checklist.csv
 no                                          aspek                                         status
  1               Tujuan dan relevansi pengumpulan                                      Terpenuhi
  2                   Legalitas sumber dan lisensi                Perlu verifikasi dokumen sumber
  3                 Transparansi sumber/provenance            Terpenuhi setelah provenance dibuat
  4                   PII (data identitas pribadi) Tidak ditemukan pada data agregat yang dipilih
  5                               Quasi-identifier                       Ada pada tingkat agregat
  6                                Minimisasi data                                      Terpenuhi
  7                      Akurasi dan kualitas data                     Diperiksa melalui validasi
  8                   Keamanan dan kepatuhan akses                

## Soal 3 & 4 — Akuisisi Data melalui API (Open-Meteo) dan Web Scraping Etis (Wikipedia)
1. **Open-Meteo Historical Weather API:** Mengambil data cuaca harian tahun 2025 (`temperature_2m_mean`, `precipitation_sum`, `wind_speed_10m_max`) untuk 6 destinasi wisata di Bali (Kuta, Tanah Lot, Ulun Danu Beratan, Ubud, GWK, Kintamani), menyimpannya sebagai `data/raw/weather_api_raw.json`, lalu mengolahnya menjadi tabel harian (`weather_api_daily.csv`) dan agregat bulanan (`weather_api_monthly.csv`).
2. **Wikipedia Ethical Scraping:** Mengunduh halaman `https://en.wikipedia.org/wiki/Bali` menggunakan *header* `User-Agent` identitas akademik dan jeda waktu (`time.sleep`), menyimpan HTML mentah ke `data/raw/wikipedia_bali_raw.html`, dan mengekstrak tabel pembagian wilayah kabupaten/kota ke `data/processed/wikipedia_bali_table.csv`.

In [4]:
# 1. Akuisisi Open-Meteo Historical Weather API
destinasi_bali = {
    "Kuta": (-8.7184, 115.1686),
    "Tanah Lot": (-8.6212, 115.0868),
    "Ulun Danu Beratan": (-8.2750, 115.1669),
    "Ubud": (-8.5069, 115.2625),
    "GWK": (-8.8105, 115.1670),
    "Kintamani": (-8.2480, 115.3540),
}

weather_raw_path = "data/raw/weather_api_raw.json"
if not os.path.exists(weather_raw_path):
    weather_raw = {}
    for nama_lokasi, (lat, lon) in destinasi_bali.items():
        params = {
            "start_date": "2025-01-01",
            "end_date": "2025-12-31",
            "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
            "timezone": "Asia/Makassar",
            "latitude": lat,
            "longitude": lon,
        }
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        weather_raw[nama_lokasi] = {"request_url": r.url, "response": r.json()}
        time.sleep(0.5)
    with open(weather_raw_path, "w", encoding="utf-8") as f:
        json.dump(weather_raw, f, indent=2)
else:
    with open(weather_raw_path, "r", encoding="utf-8") as f:
        weather_raw = json.load(f)

# Transformasi JSON mentah ke tabel harian & bulanan
daily_frames = []
for lokasi, payload in weather_raw.items():
    d = payload["response"]["daily"]
    df_loc = pd.DataFrame({
        "lokasi": lokasi,
        "tanggal": d["time"],
        "temperatur_rata2_c": d["temperature_2m_mean"],
        "curah_hujan_mm": d["precipitation_sum"],
        "kecepatan_angin_maks_kmh": d["wind_speed_10m_max"],
    })
    df_loc["periode"] = df_loc["tanggal"].str.slice(0, 7)
    daily_frames.append(df_loc)

df_daily = pd.concat(daily_frames, ignore_index=True)
df_daily.to_csv("data/processed/weather_api_daily.csv", index=False)

df_monthly = (
    df_daily.groupby(["lokasi", "periode"], as_index=False)
    .agg(
        temperatur_rata2_c=("temperatur_rata2_c", "mean"),
        curah_hujan_total_mm=("curah_hujan_mm", "sum"),
        kecepatan_angin_maks_kmh=("kecepatan_angin_maks_kmh", "max"),
    )
)
df_monthly["temperatur_rata2_c"] = df_monthly["temperatur_rata2_c"].round(2)
df_monthly["curah_hujan_total_mm"] = df_monthly["curah_hujan_total_mm"].round(2)
df_monthly["kecepatan_angin_maks_kmh"] = df_monthly["kecepatan_angin_maks_kmh"].round(2)
df_monthly.to_csv("data/processed/weather_api_monthly.csv", index=False)

# 2. Scraping Etis Wikipedia Bali
wiki_html_path = "data/raw/wikipedia_bali_raw.html"
wiki_csv_path = "data/processed/wikipedia_bali_table.csv"

if not os.path.exists(wiki_html_path):
    time.sleep(1.0)
    resp_wiki = requests.get("https://en.wikipedia.org/wiki/Bali", headers=HEADERS, timeout=30)
    resp_wiki.raise_for_status()
    with open(wiki_html_path, "w", encoding="utf-8") as f:
        f.write(resp_wiki.text)

if os.path.exists(wiki_csv_path):
    df_wiki = pd.read_csv(wiki_csv_path)
else:
    with open(wiki_html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "lxml")
    tables = pd.read_html(StringIO(str(soup)))
    df_wiki = [t for t in tables if "Kode Wilayah" in str(t.columns)][0]
    df_wiki.to_csv(wiki_csv_path, index=False)

print("=== AKUISISI OPEN-METEO HISTORICAL WEATHER API ===")
print(f"Lokasi terakuisisi (6 titik): {sorted(df_daily['lokasi'].unique().tolist())}")
print(f"Jumlah baris data harian    : {len(df_daily)} baris (365 hari x 6 lokasi)")
print(f"Jumlah baris agregat bulanan: {len(df_monthly)} baris (12 bulan x 6 lokasi)")
print(df_monthly.head(6).to_string())

print("\n=== WEB SCRAPING ETIS WIKIPEDIA BALI ===")
print(f"Tabel administratif wilayah Bali berhasil diekstrak: {df_wiki.shape[0]} baris x {df_wiki.shape[1]} kolom")
print(df_wiki[["Kode Wilayah", "Name of City or Regency", "Capital", "Area  in km2"]].head(9).to_string())

=== AKUISISI OPEN-METEO HISTORICAL WEATHER API ===
Lokasi terakuisisi (6 titik): ['GWK', 'Kintamani', 'Kuta', 'Tanah Lot', 'Ubud', 'Ulun Danu Beratan']
Jumlah baris data harian    : 2190 baris (365 hari x 6 lokasi)
Jumlah baris agregat bulanan: 72 baris (12 bulan x 6 lokasi)
  lokasi  periode  temperatur_rata2_c  curah_hujan_total_mm  kecepatan_angin_maks_kmh
0    GWK  2025-01           26.074194                 489.5                      26.3
1    GWK  2025-02           26.400000                 299.6                      34.7
2    GWK  2025-03           27.054839                 218.8                      32.8
3    GWK  2025-04           27.100000                  89.0                      21.1
4    GWK  2025-05           26.803226                 174.2                      27.3
5    GWK  2025-06           26.616667                  97.9                      31.7

=== WEB SCRAPING ETIS WIKIPEDIA BALI ===
Tabel administratif wilayah Bali berhasil diekstrak: 11 baris x 9 kolom
   Kode 

## Provenance Seluruh Sumber dan Verifikasi Integritas (MD5)
Setiap berkas mentah yang diakuisisi dicatat URL sumber, waktu akses, lisensi/ketentuan penggunaan, ukuran berkas dalam byte, dan nilai *checksum* **MD5** ke dalam `metadata/provenance.csv`.

In [5]:
def hitung_md5(file_path):
    h = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

prov_path = "metadata/provenance.csv"
df_prov = pd.read_csv(prov_path)

# Verifikasi ulang MD5 aktual dari berkas di data/raw/
df_prov["md5_verified"] = df_prov["file_name"].apply(lambda fn: hitung_md5(os.path.join("data/raw", fn)))
assert (df_prov["md5"] == df_prov["md5_verified"]).all(), "Peringatan: Hash MD5 tidak cocok!"

print("=== PROVENANCE & VERIFIKASI HASH MD5 ===")
print(df_prov[["source_name", "file_name", "file_size_bytes", "md5"]].to_string(index=False))

=== PROVENANCE & VERIFIKASI HASH MD5 ===
                             source_name               file_name  file_size_bytes                              md5
BPS - Wisatawan domestik bulanan ke Bali        dataset_bps.xlsx            38912 fb0fc90b2dcded6242f4fc5cc2cce0f3
       Open-Meteo Historical Weather API    weather_api_raw.json           160353 e1cf23c245b3f7e71cf1efc4e394dbc7
                 Wikipedia - Bali (HTML) wikipedia_bali_raw.html           938932 245aa63baeec58a300da66e77354bec4
                    Wikipedia robots.txt    wikipedia_robots.txt            28275 fc15d80fd6a4afd9f22975e740c1b48c

## Soal 5 — Penggabungan Data (`many_to_one`) dan Presentasi Hasil
Data cuaca bulanan destinasi (`weather_api_monthly.csv`, 72 baris untuk 12 bulan $	imes$ 6 lokasi) digabungkan secara *inner join* berdasarkan kunci `periode` dengan data kunjungan wisatawan domestik BPS (`bps_wisatawan_long.csv`) menggunakan validasi `many_to_one`. Hasil penggabungan menghasilkan **66 baris** (`hasil_merge_bps_weather.csv`) beserta dokumentasi keterbatasan agregasi pada `metadata/merge_metadata.json`.

In [6]:
# Siapkan agregat wisatawan per periode (unique per periode untuk relasi many_to_one)
df_bps_periode = (
    df_bps_long.sort_values(["periode", "jumlah_wisatawan"])
    .drop_duplicates(subset=["periode"], keep="first")[["periode", "jumlah_wisatawan"]]
)

# Penggabungan inner join many-to-one pada kunci 'periode'
df_merged = pd.merge(
    df_monthly,
    df_bps_periode,
    on="periode",
    how="inner",
    validate="many_to_one"
)

merge_csv_path = "data/processed/hasil_merge_bps_weather.csv"
if os.path.exists(merge_csv_path):
    df_merged = pd.read_csv(merge_csv_path)
else:
    df_merged.to_csv(merge_csv_path, index=False)

with open("metadata/merge_metadata.json", "r", encoding="utf-8") as f:
    merge_meta = json.load(f)

print("=== HASIL PENGGABUNGAN DATA (MANY-TO-ONE) ===")
print(f"Dimensi hasil_merge_bps_weather.csv : {df_merged.shape[0]} baris x {df_merged.shape[1]} kolom")
print(f"Periode tergabung ({df_merged['periode'].nunique()} bulan)         : {sorted(df_merged['periode'].unique().tolist())}")
print(f"Jumlah lokasi destinasi cuaca       : {df_merged['lokasi'].nunique()} lokasi")
print(f"Duplikasi (lokasi, periode)         : {merge_meta['duplicate_location_period']}")
print(f"Catatan Interpretasi                : {merge_meta['interpretation_note']}")
print("\nRingkasan Statistik Variabel Numerik:")
print(df_merged[["temperatur_rata2_c", "curah_hujan_total_mm", "kecepatan_angin_maks_kmh", "jumlah_wisatawan"]].describe().round(2).to_string())

=== HASIL PENGGABUNGAN DATA (MANY-TO-ONE) ===
Dimensi hasil_merge_bps_weather.csv : 66 baris x 6 kolom
Periode tergabung (11 bulan)         : ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-12']
Jumlah lokasi destinasi cuaca       : 6 lokasi
Duplikasi (lokasi, periode)         : 0
Catatan Interpretasi                : Jumlah wisatawan merupakan agregat tingkat Provinsi Bali, bukan jumlah pengunjung tiap destinasi.

Ringkasan Statistik Variabel Numerik:
       temperatur_rata2_c  curah_hujan_total_mm  kecepatan_angin_maks_kmh  jumlah_wisatawan
count               66.00                 66.00                     66.00             66.00
mean                23.90                171.43                     20.36         369822.18
std                  3.20                103.27                      6.06         154107.29
min                 17.98                 35.90                     10.80         166718.00
25%           